<a href="https://colab.research.google.com/github/imranahmed123/datascience-ai-ml/blob/main/capstone_project_sentimentanalysis_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SentimentAnalysis**



**Approach1** already trained model


(1) https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment


In [9]:
# Install dependencies
!pip install -q transformers

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch

In [11]:

# Load pretrained model and tokenizer
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [12]:
def classify_sentiment_roberta(text):
    text = text.strip().lower()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    scores = softmax(logits.numpy()[0])

    labels = ['Negative', 'Neutral', 'Positive']
    results = {label: float(score) for label, score in zip(labels, scores)}
    predicted = labels[scores.argmax()]

    return predicted, results

In [13]:
# 🔍 Test the model
text = "Howdy? Hows ur day?"
label, probs = classify_sentiment_roberta(text)
print(f"Predicted Label: {label}")
print("Confidence Scores:")
for sentiment, score in probs.items():
    print(f"{sentiment}: {score:.2f}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Predicted Label: Neutral
Confidence Scores:
Negative: 0.07
Neutral: 0.71
Positive: 0.23


**Approach2**

2) LSTM (from scratch) on Sentiment140 Dataset

In [14]:
# Install dependencies
!pip install -q nltk

In [15]:
import nltk
nltk.download('punkt_tab') # this line has been added
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [20]:
# Load dataset
try:
    df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='latin-1', header=None, on_bad_lines='skip')  # Replace 'error_bad_lines' with 'on_bad_lines'
except pd.errors.ParserError as e:
    print(f"ParserError: {e}")
    # Add more specific error handling here if needed
    df = None  # Assign a default value to df in case of an error

# Check if df was successfully loaded
if df is not None:
    df = df[[0, 5]]
    df.columns = ['label', 'text']
    df['label'] = df['label'].replace({0: 0, 2: 1, 4: 2})  # 0-Neg, 1-Neutral, 2-Pos
else:
    print("Error loading or parsing the CSV file. Please check the file path and format.")

In [21]:
# Tokenize and build vocab
tokenized_texts = df['text'].astype(str).apply(word_tokenize) # Convert the 'text' column to string type
vocab = Counter([word for tokens in tokenized_texts for word in tokens])
vocab = {word: i+2 for i, (word, _) in enumerate(vocab.most_common(10000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def encode(tokens):
    return [vocab.get(word, vocab["<UNK>"]) for word in tokens]

df['encoded'] = tokenized_texts.apply(encode)

In [22]:
# Dataset class
class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = [torch.tensor(x) for x in X]
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [23]:
def collate_fn(batch):
    x_batch, y_batch = zip(*batch)
    x_batch = pad_sequence(x_batch, batch_first=True, padding_value=0)
    y_batch = torch.tensor(y_batch)
    return x_batch, y_batch

In [24]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df['encoded'], df['label'], test_size=0.2, random_state=42)
train_dataset = SentimentDataset(X_train.tolist(), y_train.tolist())
test_dataset = SentimentDataset(X_test.tolist(), y_test.tolist())

train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate_fn)


In [25]:
# Define LSTM Model
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return out


In [26]:
# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentLSTM(len(vocab), embed_dim=64, hidden_dim=128, output_dim=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# Train
for epoch in range(3):  # Can increase
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

# Save model
torch.save(model.state_dict(), 'lstm_sentiment_model.pth')

Epoch 1 | Loss: 8820.3355
Epoch 2 | Loss: 7467.6118
Epoch 3 | Loss: 7072.0259
